In [73]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
import numpy as np
from sklearn.metrics import f1_score


In [74]:
def prepare_data_generators(data_dir, image_size=(128, 128), batch_size=32):
    train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
    train_generator = train_datagen.flow_from_directory(
        data_dir, target_size=image_size, batch_size=batch_size, class_mode='categorical', subset='training'
    )
    val_generator = train_datagen.flow_from_directory(
        data_dir, target_size=image_size, batch_size=batch_size, class_mode='categorical', subset='validation'
    )
    class_labels = list(train_generator.class_indices.keys())  # Save class labels
    return train_generator, val_generator, class_labels

In [94]:
def build_mobilenetv2_model(input_shape=(128, 128, 3), num_classes=10):
    base_model = keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    base_model.trainable = False  # Freeze base model layers
    
    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [95]:
def train_model(model, train_generator, val_generator, epochs=10):
    model.fit(train_generator, validation_data=val_generator, epochs=epochs)
    return model

In [96]:
def evaluate_f1(model, val_generator):
    y_true = []
    y_pred = []
    
    for images, labels in val_generator:
        predictions = model.predict(images)
        y_true.extend(np.argmax(labels, axis=1))
        y_pred.extend(np.argmax(predictions, axis=1))
        
        if len(y_true) >= val_generator.samples:
            break
    
    f1 = f1_score(y_true, y_pred, average='weighted')
    print(f"Validation F1 Score: {f1:.4f}")
    return f1

In [97]:
def convert_to_tflite(model, tflite_path="models/image_classifier/mobilenetv2_posture_quantized.tflite", class_labels_path="models/image_classifier/class_labels.npy"):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    np.save(class_labels_path, class_labels)
    print(f"Quantized model saved at {tflite_path}")
    print(f"Class labels saved at {class_labels_path}")

In [98]:
def load_tflite_model(tflite_path="models/image_classifier/mobilenetv2_posture_quantized.tflite", class_labels_path="models/image_classifier/class_labels.npy"):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    class_labels = np.load(class_labels_path, allow_pickle=True)
    return interpreter, class_labels.tolist()

In [99]:
def predict_image_tflite(interpreter, image_path, class_labels, image_size=(128, 128)):
    img = load_img(image_path, target_size=image_size)
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0).astype(np.float32)  # Add batch dimensionk
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    interpreter.set_tensor(input_details[0]['index'], img_array)
    interpreter.invoke()
    predictions = interpreter.get_tensor(output_details[0]['index'])
    
    predicted_class = np.argmax(predictions, axis=1)[0]
    return class_labels[predicted_class]

In [100]:
data_dir = "data/excercies_images/"  # Change this to your dataset path
train_generator, val_generator, class_labels = prepare_data_generators(data_dir)
model = build_mobilenetv2_model(num_classes=len(class_labels))
model = train_model(model, train_generator, val_generator)
f1 = evaluate_f1(model, val_generator)
convert_to_tflite(model)
    
    # Load TFLite model and test prediction


Found 11090 images belonging to 22 classes.
Found 2763 images belonging to 22 classes.
Epoch 1/10


D:\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


347/347 ━━━━━━━━━━━━━━━━━━━━ 104s 286ms/step - accuracy: 0.4752 - loss: 1.8706 - val_accuracy: 0.6352 - val_loss: 1.2845
Epoch 2/10
347/347 ━━━━━━━━━━━━━━━━━━━━ 98s 282ms/step - accuracy: 0.8908 - loss: 0.4026 - val_accuracy: 0.6924 - val_loss: 1.0907
Epoch 3/10
347/347 ━━━━━━━━━━━━━━━━━━━━ 98s 283ms/step - accuracy: 0.9343 - loss: 0.2348 - val_accuracy: 0.7072 - val_loss: 1.1085
Epoch 4/10
347/347 ━━━━━━━━━━━━━━━━━━━━ 99s 286ms/step - accuracy: 0.9465 - loss: 0.1857 - val_accuracy: 0.7072 - val_loss: 1.1890
Epoch 5/10
347/347 ━━━━━━━━━━━━━━━━━━━━ 98s 284ms/step - accuracy: 0.9584 - loss: 0.1405 - val_accuracy: 0.7141 - val_loss: 1.1525
Epoch 6/10
347/347 ━━━━━━━━━━━━━━━━━━━━ 99s 284ms/step - accuracy: 0.9686 - loss: 0.1068 - val_accuracy: 0.7257 - val_loss: 1.1563
Epoch 7/10
347/347 ━━━━━━━━━━━━━━━━━━━━ 98s 282ms/step - accuracy: 0.9721 - loss: 0.0988 - val_accuracy: 0.7181 - val_loss: 1.2892
Epoch 8/10
347/347 ━━━━━━━━━━━━━━━━━━━━ 98s 282ms/step - accuracy: 0.9734 - loss: 0.0820 - va

INFO:tensorflow:Assets written to: C:\Users\HP\AppData\Local\Temp\tmp6qmxmxm5\assets


Saved artifact at 'C:\Users\HP\AppData\Local\Temp\tmp6qmxmxm5'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor_1754')
Output Type:
  TensorSpec(shape=(None, 22), dtype=tf.float32, name=None)
Captures:
  2196952740304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2196952738384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2196952738000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2196952736848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2196952737808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2196952735696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2196952728208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2196952734928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2196952731088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2196952737616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  21

In [105]:
interpreter, class_labels = load_tflite_model()
test_image_path = "data/excercies_images/pull up/pull up_100061.jpg"  # Change this to an actual image path
predicted_label = predict_image_tflite(interpreter=interpreter,image_path=test_image_path, class_labels=class_labels)
print(f"Predicted class: {predicted_label}")

Predicted class: pull up
